# AnemoScan: Ai - Based Early Detection Of Anemia From Nail & Conjunctiva Images
## YOLOv8 Object Detection (Conjunctiva & Nails)
- This script uses the Ultralytics YOLOv8 library to train a custom object detection model (e.g., for detecting signs of anemia in nail or conjunctiva images) and then runs inference on a test image. The pipeline includes loading a pre‑trained model, fine‑tuning on a custom dataset, and visualising predictions.

## 1. Import Required Library
- We import the (YOLO) class from the (ultralytics) package, which provides a high‑level interface for loading models, training, and inference.

In [1]:
import os
import cv2
import shutil
import random
import numpy as np
from ultralytics import YOLO
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

## 2. Rename Dataset Images
- In this section, all images are renamed systematically to ensure consistency and avoid duplication. Each image is assigned a unique name based on its category (conjunctiva or nails) and an incremental index.

In [3]:
def rename_images(base_path):
    folders = ['Conjunctiva', 'Nails']

    for folder in folders:
        folder_path = os.path.join(base_path, folder)
        files = sorted(os.listdir(folder_path))

        count = 1

        for file in files:
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            old_path = os.path.join(folder_path, file)

            if folder.lower() == 'conjunctiva':
                prefix = 'conj'
            else:
                prefix = 'nail'

            new_name = f'{prefix}_{count:05d}.jpg'
            new_path = os.path.join(folder_path, new_name)

            os.rename(old_path, new_path)
            count += 1

    print('Renaming completed successfully!')

base_path = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection'
rename_images(base_path)

Renaming completed successfully!


## 3. Create Dataset Structure & Split Dataset into Train, Validation, and Test Sets
 - In this section, the required folder structure for training, validation, and testing is created. Each split contains separate folders for conjunctiva and nails images to support classification.
 - In this section, the dataset is split into training, validation, and testing sets using predefined ratios. This ensures proper model evaluation and prevents data leakage.

In [5]:
# Path to dataset folder
dataset_path = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8'

# Where current train images & labels are
images_dir = os.path.join(dataset_path, 'train', 'images')
labels_dir = os.path.join(dataset_path, 'train', 'labels')

# Get all image filenames (without extension)
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
image_names = [os.path.splitext(f)[0] for f in image_files]

# Split into train (70%), val (15%), test (15%)
train_names, test_names = train_test_split(image_names, test_size = 0.3, random_state = 42)
val_names, test_names = train_test_split(test_names, test_size = 0.5, random_state = 42)

# Create destination folders
for split in ['val', 'test']:
    os.makedirs(os.path.join(dataset_path, split, 'images'), exist_ok = True)
    os.makedirs(os.path.join(dataset_path, split, 'labels'), exist_ok = True)

def move_files(name_list, split):
    for name in name_list:
        # Find the image file (handles .jpg or .png)
        img_src = None
        for ext in ['.jpg', '.png', '.jpeg']:
            candidate = os.path.join(images_dir, name + ext)
            if os.path.exists(candidate):
                img_src = candidate
                break
        if img_src is None:
            print(f'Warning: Image {name} not found')
            continue

        # Move image
        dst_img = os.path.join(dataset_path, split, 'images', os.path.basename(img_src))
        shutil.move(img_src, dst_img)

        # Move corresponding label
        lbl_src = os.path.join(labels_dir, name + '.txt')
        if os.path.exists(lbl_src):
            dst_lbl = os.path.join(dataset_path, split, 'labels', name + '.txt')
            shutil.move(lbl_src, dst_lbl)
        else:
            print(f'Warning: No label for {name}')

# Move validation and test files out of the train folder
move_files(val_names, 'val')
move_files(test_names, 'test')

print(f'✅ Done! Remaining training images: {len(os.listdir(images_dir))}')
print(f'Validation images: {len(os.listdir(os.path.join(dataset_path, 'val', 'images')))}')
print(f'Test images: {len(os.listdir(os.path.join(dataset_path, 'test', 'images')))}')

✅ Done! Remaining training images: 1311
Validation images: 281
Test images: 282


## 4. Load Pre‑trained YOLOv8 Model
- A pre‑trained YOLOv8 Nano model (yolov8n.pt) is loaded. This model has been trained on the COCO dataset and serves as a starting point for transfer learning on our custom anemia‑related dataset.

In [7]:
model = YOLO('yolov8n.pt')

## 5. Train the Model on Custom Data
- The model is fine‑tuned using a custom dataset defined in data.yaml (which contains paths to training / validation images and class names).

In [9]:
model.train(
    data = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\data.yaml',
    epochs = 50,
    imgsz = 640,
    batch = 16,
    patience = 10,
    project = str(r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8'),
    name = 'anemoscan_roi',
    exist_ok = True,
    save = True
)

Ultralytics 8.4.46  Python-3.12.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=anemoscan_roi, nbs=64, nms=False, opse

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000029E27F4FEC0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

## 6. Load the Best Trained Model
- Once training is complete, we load the newly trained model weights from (best.pt) for inference.

In [11]:
best_model_path = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8/anemoscan_roi/weights/best.pt'
model = YOLO(best_model_path)

## 7. Run Inference on a Test Image
- The model is applied to a single test image (test_image.jpg).
- The results object contains detection information: bounding boxes, confidence scores, class labels, and optionally masks.

In [15]:
metrics = model.val(data = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\data.yaml', split = 'test')

print(f'Test mAP50: {metrics.box.map50:.4f}')
print(f'Test mAP50-95: {metrics.box.map:.4f}')

Ultralytics 8.4.46  Python-3.12.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
val: Fast image access  (ping: 0.20.1 ms, read: 438.6853.2 MB/s, size: 1635.0 KB)
val: Scanning D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\test\labels.cache... 282 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 282/282  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.4s/it 43.5s2.3ss
                   all        282        297      0.995      0.998      0.994      0.761
           conjunctiva         81         81      0.999          1      0.995      0.734
                  nail        201        216       0.99      0.995      0.994      0.789
Speed: 1.1ms preprocess, 74.4ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Users\20109\runs\detect\val-2
Test mAP50: 0.9943
Test mAP50-95: 0.7612


In [17]:
results = model(r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Conjunctiva.jpeg', conf = 0.5)

results[0].show()
results[0].save('my_conjunctiva_prediction.jpg')


image 1/1 D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Conjunctiva.jpeg: 640x480 1 conjunctiva, 144.5ms
Speed: 4.6ms preprocess, 144.5ms inference, 7.8ms postprocess per image at shape (1, 3, 640, 480)


'my_conjunctiva_prediction.jpg'

In [19]:
results = model(r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Nail.jpeg', conf = 0.5)

results[0].show()
results[0].save('my_nail_prediction.jpg')


image 1/1 D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Nail.jpeg: 640x480 4 nails, 101.7ms
Speed: 3.5ms preprocess, 101.7ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 480)


'my_nail_prediction.jpg'

## 8. Export the Model for Deployment
- This snippet exports a trained YOLO model to the ONNX (Open Neural Network Exchange) format, which is optimized for fast and portable inference—especially on CPU.

In [21]:
# Export to ONNX (fast CPU inference)
model.export(format = 'onnx')

Ultralytics 8.4.46  Python-3.12.4 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\anemoscan_roi\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB 1.3 MB/s eta 0:00:13
   ---------------------------------------- 0.1/16.4 MB 1.2 MB/s eta 0:00:14
    --------------------------------------- 0.2/16.4 MB 1.6 MB/s eta 0:00:11
    --------------------------------------- 0.3/16.4 MB 2.0 MB/s eta 0:00:09
   - ------------------------------------

'D:\\Spring 2025-2026\\TM471B\\AnemoScan\\Data\\Detection\\AnemoScan ROI Detection.yolov8\\anemoscan_roi\\weights\\best.onnx'

## 9. ROI Extraction
- This script extracts Regions of Interest (ROIs) from an input image using a trained YOLO model. Each detected object is cropped and saved as a separate image file.

In [23]:
def extract_roi(image_path, model, output_folder = r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\rois'):
    
    os.makedirs(output_folder, exist_ok = True)
    
    results = model(image_path, conf = 0.5)
    img = cv2.imread(image_path)
    
    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())  # box corners
        cls = int(box.cls[0])
        class_name = model.names[cls]
        
        roi = img[y1 : y2, x1 : x2]
        out_path = os.path.join(output_folder, f'{class_name}_{i}.jpg')
        cv2.imwrite(out_path, roi)
        print(f'Saved {out_path}')
    
    return results

extract_roi(r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Conjunctiva.jpeg', model)
extract_roi(r'D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Nail.jpeg', model)


image 1/1 D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Conjunctiva.jpeg: 640x480 1 conjunctiva, 99.6ms
Speed: 3.4ms preprocess, 99.6ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 480)
Saved rois\conjunctiva_0.jpg

image 1/1 D:\Spring 2025-2026\TM471B\AnemoScan\Data\Detection\AnemoScan ROI Detection.yolov8\My_Nail.jpeg: 640x480 4 nails, 73.1ms
Speed: 2.8ms preprocess, 73.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 480)
Saved rois\nail_0.jpg
Saved rois\nail_1.jpg
Saved rois\nail_2.jpg
Saved rois\nail_3.jpg


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'conjunctiva', 1: 'nail'}
 obb: None
 orig_img: array([[[ 37,  46,  49],
         [ 44,  53,  56],
         [ 52,  63,  67],
         ...,
         [ 62,  82,  99],
         [ 61,  84, 100],
         [ 63,  86, 102]],
 
        [[ 49,  56,  59],
         [ 50,  57,  60],
         [ 48,  57,  60],
         ...,
         [ 67,  90, 106],
         [ 69,  92, 108],
         [ 69,  92, 108]],
 
        [[ 62,  67,  68],
         [ 56,  61,  62],
         [ 47,  52,  53],
         ...,
         [ 71,  93, 111],
         [ 74,  97, 113],
         [ 71,  96, 112]],
 
        ...,
 
        [[140, 156, 192],
         [140, 156, 192],
         [140, 156, 192],
         ...,
         [ 82,  95, 117],
         [ 91, 104, 126],
         [ 86,  99, 121]],
 
        [[140, 156, 192],
         [140, 156, 192],
         [139, 155, 191],
         ...,
  